In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error, mean_squared_error

# Carregar os dados
arquivo = 'guias_de_itbi_pagas_2026.xlsx'
xls = pd.ExcelFile(arquivo)

# Filtrar apenas as abas referentes aos meses
abas_meses = [aba for aba in xls.sheet_names if '-' in aba]

# Ler e concatenar todas as abas de meses num único DataFrame
df_list = [pd.read_excel(arquivo, sheet_name=aba) for aba in abas_meses]
df = pd.concat(df_list, ignore_index=True)

# Converter a coluna alvo para numérico
target = 'Valor de Transação (declarado pelo contribuinte)'
df[target] = pd.to_numeric(df[target], errors='coerce')

# Definir as variáveis numéricas e de texto
features_numericas = ['Valor Venal de Referência', 'Área Construída (m2)', 'Área do Terreno (m2)']
features_categoricas = ['Situação do SQL']

features = features_numericas + features_categoricas

# Filtrar e limpar o dataset
df_model = df[features + [target]].dropna()
df_model = df_model[df_model[target] > 1000]

X = df_model[features]
y = df_model[target]

# Aplicar o One-Hot Encoding nas colunas de texto
X = pd.get_dummies(X, columns=features_categoricas, drop_first=True)

# Divisão em Treinamento e Teste (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

# Treinamento do Modelo
modelo = RandomForestRegressor(n_estimators=100, random_state=123)
modelo.fit(X_train, y_train)

# Avaliação do Modelo
previsoes = modelo.predict(X_test)

mae = mean_absolute_error(y_test, previsoes)
rmse = np.sqrt(mean_squared_error(y_test, previsoes))
mape = mean_absolute_percentage_error(y_test, previsoes)
r2 = r2_score(y_test, previsoes)

# Exibindo os resultados
print(f"Tamanho do conjunto de treinamento: {len(X_train)} registros")
print(f"Tamanho do conjunto de teste: {len(X_test)} registros\n")

print("--- RESULTADOS DA AVALIAÇÃO ---")
print(f"R² (Coeficiente de Determinação): {r2:.4f} (Mais próximo de 1.0 é melhor)")
print(f"Erro Percentual (MAPE): {mape * 100:.2f}% (O modelo erra em média {mape * 100:.2f}% do valor real)")
print(f"Erro Médio Absoluto (MAE): R$ {mae:,.2f} (O modelo erra em média R$ {mae:,.2f} para mais ou para menos)")
print(f"Raiz do Erro Quadrático Médio (RMSE): R$ {rmse:,.2f} (Penaliza erros muito grandes)")

Tamanho do conjunto de treinamento: 124040 registros
Tamanho do conjunto de teste: 31010 registros

--- RESULTADOS DA AVALIAÇÃO ---
R² (Coeficiente de Determinação): 0.4760 (Mais próximo de 1.0 é melhor)
Erro Percentual (MAPE): 283.78% (O modelo erra em média 283.78% do valor real)
Erro Médio Absoluto (MAE): R$ 267,811.87 (O modelo erra em média R$ 267,811.87 para mais ou para menos)
Raiz do Erro Quadrático Médio (RMSE): R$ 2,466,249.76 (Penaliza erros muito grandes)
